# Assignment: Building a Neural Network from Scratch
## Boston/California Housing Price Prediction | Pure Python & NumPy

---

### Assignment Objectives

By completing this assignment you will:

1. **Explain** the core components of a neural network: neurons, layers, weights, biases, and activation functions.
2. **Implement** a complete neural network from scratch using only Python and NumPy — no frameworks allowed.
3. **Implement** the MSE loss function and its gradient for regression tasks.
4. **Derive and implement** backpropagation using the chain rule to compute gradients layer by layer.
5. **Train** your from-scratch network on the California Housing dataset and visualize the training process.
6. **Experiment** with architectures, learning rates, and activation functions, documenting the effect of each change.

> **Deliverable:** After running every cell, you will have all the figures, tables, and analysis needed to write a professional lab report.

---

### Report Structure Guide

Your report should follow this structure (each section maps to a part of this notebook):

| Report Section | What to Write | Notebook Section |
|---|---|---|
| **1. Introduction** | What are neural networks? Neurons, layers, activations, why non-linearity? | Part 1 |
| **2. Dataset** | Describe California Housing: features, size, distributions, preprocessing | Part 3 |
| **3. Methodology** | Architecture, forward pass, loss function, gradient descent, backprop | Parts 4-10 |
| **4. Training** | Training procedure, epochs, batch size, learning rate, loss curves | Part 11 |
| **5. Results** | Test metrics (MSE, RMSE, MAE, R²), predicted vs actual plots | Part 12 |
| **6. Experiments** | Architecture / LR / activation experiments with comparison plots | Part 13 |
| **7. Conclusion** | Summary of findings, what you learned, potential improvements | Part 14 |

### Grading Rubric

| Category | Points | Description |
|---|---|---|
| Architecture & Forward Pass | 20 | Correct neuron, layer, and MLP implementations with proper matrix dimensions |
| Loss Functions | 15 | MSE implementation, gradient derivation, initial loss analysis |
| Gradient Descent | 15 | 1D optimization demo, learning rate effect, convergence analysis |
| Backpropagation | 20 | Chain rule derivation, backward pass, gradient checking verification |
| Training & Evaluation | 20 | Training loop, loss curves, test metrics, prediction analysis |
| Experiments | 10 | Systematic experiments with comparison plots and analysis |
| **Total** | **100** | |

---

*Apeiron AI | "Boundless Possibilities, Infinite Potential"*
*© 2026 | www.aperionaiml.com*

---

# Part 1: Background — What is a Neural Network?

## 1.1 The Neuron

The fundamental unit of a neural network is the **neuron** (also called a *node* or *unit*). A neuron takes multiple inputs, computes a weighted sum, adds a bias, and applies an activation function:

```
        x₁ ──w₁──┐
                  │
        x₂ ──w₂──┼──┐ z = w·x + b │    a = f(z)
                  │  └────────────┼────────────► output
        x₃ ──w₃──┘  ┌────────────┘
                   b (bias)
```

**output = f(w·x + b)**

Where:
- **x** = input vector (features of one data point)
- **w** = weight vector (learned parameters)
- **b** = bias (learned scalar)
- **f** = activation function (introduces non-linearity)

## 1.2 Layers and Matrix Multiplication

A **layer** groups multiple neurons together. Each neuron in the layer receives ALL inputs and produces one output:

```
                         Layer (4 neurons)
                      ┌───────────────────┐
   x₁ ─────────┼── neuron₁ ────┼── a₁
                      │                   │
   x₂ ─────────┼── neuron₂ ────┼── a₂
                      │                   │
   x₃ ─────────┼── neuron₃ ────┼── a₃
                      │                   │
                      ┼── neuron₄ ────┼── a₄
                      └───────────────────┘
```

In matrix form: **a = f(W · x + b)**

Where W is a (4 × 3) weight matrix — each row is one neuron's weights.

## 1.3 Multi-Layer Perceptron (MLP)

Stack multiple layers to form a **Multi-Layer Perceptron**:

```
   Input (8)        Hidden₁ (64)      Hidden₂ (32)       Output (1)
  ┌────────┐     ┌──────────┐     ┌──────────┐     ┌────────┐
  │ MedInc  │────►│          │────►│          │────►│        │
  │ HouseAge│     │  ReLU    │     │  ReLU    │     │ Price  │
  │ AveRooms│     │          │     │          │     │ (linear│
  │ ...     │     │ 64       │     │ 32       │     │ output)│
  │ (8 feat)│     │ neurons  │     │ neurons  │     │        │
  └────────┘     └──────────┘     └──────────┘     └────────┘

  Data flows left → right (FORWARD PASS)
```

## 1.4 Why Non-Linearity Matters

Without activation functions, stacking layers is pointless:

**Layer₂(Layer₁(x)) = W₂(W₁x + b₁) + b₂ = (W₂W₁)x + (W₂b₁ + b₂) = W'x + b'**

Two linear layers collapse into ONE linear layer! Non-linear activations (ReLU, sigmoid) break this collapse and let the network learn complex patterns.

## 1.5 Key Equations

| Concept | Equation | Description |
|---|---|---|
| **ReLU** | f(z) = max(0, z) | Zero out negatives, pass positives |
| **Sigmoid** | f(z) = 1 / (1 + e⁻ᶻ) | Squash to (0, 1) |
| **MSE Loss** | L = (1/n) ∑ (ŷ - y)² | Mean Squared Error for regression |
| **Gradient Descent** | w = w - α · ∂L/∂w | Update weights in direction that reduces loss |
| **Chain Rule** | ∂L/∂w = ∂L/∂a · ∂a/∂z · ∂z/∂w | Backpropagation — compute gradients layer by layer |

## 1.6 The Training Process

1. **Forward pass**: compute predictions from inputs
2. **Loss**: measure how wrong the predictions are
3. **Backward pass (backprop)**: compute gradients of the loss w.r.t. every weight
4. **Update**: adjust weights using gradient descent
5. **Repeat** for many epochs until the loss converges

---

# Part 2: Environment Setup

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  INSTALL DEPENDENCIES (run once)                                            ║
# ╚══════════════════════════════════════════════════════════════════════════════╝
%pip install numpy matplotlib pandas scikit-learn --quiet


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  IMPORTS                                                                    ║
# ╚══════════════════════════════════════════════════════════════════════════════╝
# IMPORTANT: This assignment uses ONLY Python + NumPy.
# No PyTorch, TensorFlow, or any deep learning framework allowed.

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# scikit-learn is used ONLY for loading the dataset
from sklearn.datasets import fetch_california_housing

# ── Plotting defaults ────────────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.dpi": 120,
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
})

# ── Reproducibility ──────────────────────────────────────────────────────────────
SEED = 42
np.random.seed(SEED)
print(f"NumPy version   : {np.__version__}")
print(f"Random seed     : {SEED}")
print(f"Framework       : Pure Python + NumPy (NO PyTorch)")


---

# Part 3: California Housing Dataset

## 3.1 About the Dataset

The **California Housing** dataset contains 20,640 samples of California census block groups from the 1990 census. Each sample has **8 numerical features** and the target is the **median house value** (in $100,000s).

| Feature | Description |
|---|---|
| MedInc | Median income in block group |
| HouseAge | Median house age in block group |
| AveRooms | Average number of rooms per household |
| AveBedrms | Average number of bedrooms per household |
| Population | Block group population |
| AveOccup | Average number of household members |
| Latitude | Block group latitude |
| Longitude | Block group longitude |

**Target**: Median house value (in $100,000s), ranging from 0.15 to 5.0

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  LOAD CALIFORNIA HOUSING DATASET                                            ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

housing = fetch_california_housing()
X_raw = housing.data          # (20640, 8)
y_raw = housing.target        # (20640,)
feature_names = housing.feature_names

df = pd.DataFrame(X_raw, columns=feature_names)
df["MedHouseVal"] = y_raw

print(f"Dataset shape: {X_raw.shape}")
print(f"Target shape:  {y_raw.shape}")
print(f"Features: {feature_names}")
print()
df.head(10)


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  STATISTICAL SUMMARY                                                        ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

df.describe().round(3)


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  VISUALIZE FEATURE DISTRIBUTIONS                                            ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

fig, axes = plt.subplots(3, 3, figsize=(14, 10))
axes = axes.flatten()

for i, col in enumerate(df.columns):
    axes[i].hist(df[col], bins=50, alpha=0.7, color="steelblue", edgecolor="white")
    axes[i].set_title(col, fontweight="bold")
    axes[i].set_xlabel("Value")
    axes[i].set_ylabel("Count")

plt.suptitle("Feature Distributions — California Housing", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.savefig("asgn_fig_feature_distributions.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CORRELATION WITH TARGET                                                    ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, feat in enumerate(feature_names):
    axes[i].scatter(X_raw[:, i], y_raw, alpha=0.05, s=1, color="steelblue")
    axes[i].set_xlabel(feat)
    axes[i].set_ylabel("MedHouseVal")
    corr = np.corrcoef(X_raw[:, i], y_raw)[0, 1]
    axes[i].set_title(f"{feat} (r={corr:.3f})", fontweight="bold")

plt.suptitle("Feature vs Target Correlations", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.savefig("asgn_fig_correlations.png", dpi=150, bbox_inches="tight")
plt.show()


## 3.2 Preprocessing: Normalization from Scratch

We normalize each feature to have **zero mean** and **unit variance**: x_norm = (x - mean) / std

This is done **from scratch** — no sklearn StandardScaler allowed.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  NORMALIZE & SPLIT (from scratch — no sklearn)                              ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

# --- Normalize features (z-score) from scratch ---
X_mean = np.mean(X_raw, axis=0)
X_std  = np.std(X_raw, axis=0)
X_norm = (X_raw - X_mean) / X_std

# --- Normalize target ---
y_mean = np.mean(y_raw)
y_std  = np.std(y_raw)
y_norm = (y_raw - y_mean) / y_std

print("Feature means (should be ~0):", np.mean(X_norm, axis=0).round(6))
print("Feature stds  (should be ~1):", np.std(X_norm, axis=0).round(6))

# --- Train/test split 80/20 (from scratch) ---
n = X_norm.shape[0]
indices = np.random.permutation(n)
split = int(0.8 * n)

X_train = X_norm[indices[:split]]
X_test  = X_norm[indices[split:]]
y_train = y_norm[indices[:split]].reshape(-1, 1)
y_test  = y_norm[indices[split:]].reshape(-1, 1)

print(f"\nTraining set: X={X_train.shape}, y={y_train.shape}")
print(f"Test set:     X={X_test.shape}, y={y_test.shape}")


---

# Part 4: Activation Functions (from Scratch)

Every activation function and its derivative is implemented using only NumPy.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  ACTIVATION FUNCTIONS FROM SCRATCH                                          ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

def relu(z):
    """ReLU: f(z) = max(0, z)"""
    return np.maximum(0, z)

def relu_derivative(z):
    """ReLU derivative: 1 if z > 0, else 0"""
    return (z > 0).astype(float)

def sigmoid(z):
    """Sigmoid: f(z) = 1 / (1 + exp(-z))"""
    z_clipped = np.clip(z, -500, 500)  # prevent overflow
    return 1.0 / (1.0 + np.exp(-z_clipped))

def sigmoid_derivative(z):
    """Sigmoid derivative: sigmoid(z) * (1 - sigmoid(z))"""
    s = sigmoid(z)
    return s * (1.0 - s)

def tanh_act(z):
    """Tanh activation"""
    return np.tanh(z)

def tanh_derivative(z):
    """Tanh derivative: 1 - tanh(z)^2"""
    return 1.0 - np.tanh(z) ** 2

def linear(z):
    """Linear (identity) activation for output layer"""
    return z

def linear_derivative(z):
    """Linear derivative: always 1"""
    return np.ones_like(z)

# --- Test on sample values ---
z_test = np.array([-2.0, -1.0, 0.0, 1.0, 2.0])
print("z values:        ", z_test)
print("ReLU(z):         ", relu(z_test))
print("ReLU'(z):        ", relu_derivative(z_test))
print("Sigmoid(z):      ", sigmoid(z_test).round(4))
print("Sigmoid'(z):     ", sigmoid_derivative(z_test).round(4))


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  PLOT ACTIVATIONS AND DERIVATIVES                                           ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

z = np.linspace(-5, 5, 300)
activations = [
    ("ReLU", relu(z), relu_derivative(z)),
    ("Sigmoid", sigmoid(z), sigmoid_derivative(z)),
    ("Tanh", tanh_act(z), tanh_derivative(z)),
]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, act, deriv) in zip(axes, activations):
    ax.plot(z, act, "b-", linewidth=2, label=f"{name}(z)")
    ax.plot(z, deriv, "r--", linewidth=2, label=f"{name}'(z)")
    ax.axhline(0, color="gray", linewidth=0.5)
    ax.axvline(0, color="gray", linewidth=0.5)
    ax.set_title(name, fontweight="bold")
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle("Activation Functions and Their Derivatives", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("asgn_fig_activations.png", dpi=150, bbox_inches="tight")
plt.show()


> **For your report:** Explain why ReLU is preferred over sigmoid for hidden layers. Consider: gradient magnitude, computational cost, and the vanishing gradient problem.

---

# Part 5: A Single Neuron

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  SINGLE NEURON FROM SCRATCH                                                 ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

def single_neuron(x, w, b, activation=relu):
    """
    A single neuron: output = activation(w . x + b)

    Args:
        x: input vector (n_features,)
        w: weight vector (n_features,)
        b: bias scalar
        activation: activation function
    Returns:
        z: pre-activation (weighted sum + bias)
        a: post-activation output
    """
    z = np.dot(w, x) + b
    a = activation(z)
    return z, a

# --- Test on one house ---
x_sample = X_train[0]  # one house, 8 features
w = np.random.randn(8) * 0.1
b = 0.0

z, a = single_neuron(x_sample, w, b)

print("Input features (normalized):")
for name, val in zip(feature_names, x_sample):
    print(f"  {name:12s}: {val:+.4f}")
print(f"\nWeighted sum (z): {z:.4f}")
print(f"Bias (b):          {b:.4f}")
print(f"After ReLU (a):    {a:.4f}")


> **For your report:** What does the neuron compute? Explain in terms of a weighted vote of the input features.

---

# Part 6: Dense Layer

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  DENSE LAYER FROM SCRATCH                                                   ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

def dense_layer(X, W, b, activation=relu):
    """
    A dense (fully-connected) layer: a = activation(X @ W.T + b)

    Args:
        X: input matrix (batch_size, n_in)
        W: weight matrix (n_out, n_in)
        b: bias vector (n_out,)
        activation: activation function
    Returns:
        z: pre-activation (batch_size, n_out)
        a: post-activation (batch_size, n_out)
    """
    z = X @ W.T + b    # (batch, n_in) @ (n_in, n_out) + (n_out,)
    a = activation(z)
    return z, a

# --- Test: 8 inputs -> 4 neurons ---
n_in, n_out = 8, 4
W = np.random.randn(n_out, n_in) * np.sqrt(2.0 / n_in)  # He init
b = np.zeros(n_out)

X_batch = X_train[:5]  # 5 samples
z, a = dense_layer(X_batch, W, b)

print(f"Input shape:  X = {X_batch.shape}   (batch_size=5, features=8)")
print(f"Weight shape: W = {W.shape}        (neurons=4, features=8)")
print(f"Bias shape:   b = {b.shape}           (neurons=4,)")
print(f"Output shape: a = {a.shape}        (batch_size=5, neurons=4)")
print(f"\nFirst sample output: {a[0].round(4)}")


> **For your report:** Explain the matrix dimensions. Why is W shaped (n_out, n_in)? How does matrix multiplication replace looping over neurons?

---

# Part 7: Complete MLP Class

This is the core of the assignment: a full neural network implemented from scratch.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  NEURAL NETWORK CLASS (from scratch)                                        ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

class NeuralNetwork:
    """
    Multi-Layer Perceptron implemented from scratch with NumPy.

    Supports:
    - Arbitrary layer sizes (e.g., [8, 64, 32, 1])
    - He weight initialization
    - ReLU / Sigmoid / Tanh activations for hidden layers
    - Linear output for regression
    - MSE loss
    - Full backpropagation with chain rule
    - Mini-batch SGD
    """

    ACTIVATIONS = {
        "relu":    (relu, relu_derivative),
        "sigmoid": (sigmoid, sigmoid_derivative),
        "tanh":    (tanh_act, tanh_derivative),
        "linear":  (linear, linear_derivative),
    }

    def __init__(self, layer_sizes, activation="relu"):
        """
        Args:
            layer_sizes: list of ints, e.g. [8, 64, 32, 1]
            activation: activation for hidden layers ("relu", "sigmoid", "tanh")
        """
        self.layer_sizes = layer_sizes
        self.n_layers = len(layer_sizes) - 1  # number of weight matrices
        self.act_name = activation
        self.act_fn, self.act_deriv = self.ACTIVATIONS[activation]

        # --- He initialization ---
        self.weights = []
        self.biases = []
        for i in range(self.n_layers):
            n_in = layer_sizes[i]
            n_out = layer_sizes[i + 1]
            # He init: W ~ N(0, sqrt(2/n_in))
            W = np.random.randn(n_out, n_in) * np.sqrt(2.0 / n_in)
            b = np.zeros(n_out)
            self.weights.append(W)
            self.biases.append(b)

        # Storage for forward/backward
        self.z_cache = []   # pre-activations
        self.a_cache = []   # post-activations

    def forward(self, X):
        """
        Forward pass: compute output for input X.
        Stores intermediate values for backprop.

        Args:
            X: input (batch_size, n_features)
        Returns:
            output: predictions (batch_size, 1)
        """
        self.z_cache = []
        self.a_cache = [X]  # input is the "activation" of layer 0

        a = X
        for i in range(self.n_layers):
            z = a @ self.weights[i].T + self.biases[i]
            self.z_cache.append(z)

            if i < self.n_layers - 1:
                # Hidden layer: use activation
                a = self.act_fn(z)
            else:
                # Output layer: linear (regression)
                a = z

            self.a_cache.append(a)

        return a

    def compute_loss(self, y_pred, y_true):
        """MSE loss: L = (1/n) * sum((y_pred - y_true)^2)"""
        return np.mean((y_pred - y_true) ** 2)

    def backward(self, y_true):
        """
        Backpropagation: compute gradients dW and db for each layer.
        Uses the chain rule applied layer by layer from output to input.

        Returns:
            dW_list: list of weight gradients
            db_list: list of bias gradients
        """
        m = y_true.shape[0]  # batch size

        dW_list = [None] * self.n_layers
        db_list = [None] * self.n_layers

        # Output layer gradient: dL/dz_out = (2/m) * (y_pred - y_true)
        y_pred = self.a_cache[-1]
        dz = (2.0 / m) * (y_pred - y_true)  # (batch, 1)

        for i in reversed(range(self.n_layers)):
            a_prev = self.a_cache[i]  # activation from previous layer

            # dW = dz.T @ a_prev / m  (averaged over batch)
            dW_list[i] = dz.T @ a_prev / m
            # db = mean of dz over batch
            db_list[i] = np.mean(dz, axis=0)

            if i > 0:
                # Propagate gradient to previous layer
                # da_prev = dz @ W
                da = dz @ self.weights[i]
                # dz_prev = da * activation_derivative(z_prev)
                dz = da * self.act_deriv(self.z_cache[i - 1])

        return dW_list, db_list

    def update(self, dW_list, db_list, lr):
        """SGD update: W -= lr * dW, b -= lr * db"""
        for i in range(self.n_layers):
            self.weights[i] -= lr * dW_list[i]
            self.biases[i]  -= lr * db_list[i]

    def train(self, X_tr, y_tr, X_val, y_val, lr=0.01, epochs=200,
              batch_size=64, verbose=True):
        """
        Mini-batch SGD training loop.

        Returns:
            history: dict with 'train_loss' and 'val_loss' per epoch
        """
        history = {"train_loss": [], "val_loss": []}
        n = X_tr.shape[0]

        for epoch in range(epochs):
            # Shuffle training data
            idx = np.random.permutation(n)
            X_shuffled = X_tr[idx]
            y_shuffled = y_tr[idx]

            epoch_loss = 0.0
            n_batches = 0

            for start in range(0, n, batch_size):
                end = min(start + batch_size, n)
                X_batch = X_shuffled[start:end]
                y_batch = y_shuffled[start:end]

                # Forward
                y_pred = self.forward(X_batch)
                batch_loss = self.compute_loss(y_pred, y_batch)
                epoch_loss += batch_loss
                n_batches += 1

                # Backward
                dW, db = self.backward(y_batch)

                # Update
                self.update(dW, db, lr)

            # Record losses
            train_loss = epoch_loss / n_batches
            val_pred = self.forward(X_val)
            val_loss = self.compute_loss(val_pred, y_val)

            history["train_loss"].append(train_loss)
            history["val_loss"].append(val_loss)

            if verbose and (epoch + 1) % 20 == 0:
                print(f"Epoch {epoch+1:4d}/{epochs}  |  "
                      f"Train Loss: {train_loss:.6f}  |  Val Loss: {val_loss:.6f}")

        return history

    def summary(self):
        """Print a summary of the network architecture."""
        total_params = 0
        print("=" * 55)
        print(f"  Neural Network Summary (from scratch)")
        print("=" * 55)
        print(f"  {'Layer':<12} {'Shape':<20} {'Parameters':<12}")
        print("-" * 55)
        for i in range(self.n_layers):
            w_shape = self.weights[i].shape
            n_params = w_shape[0] * w_shape[1] + w_shape[0]
            act = self.act_name if i < self.n_layers - 1 else "linear"
            print(f"  Layer {i+1:<5} {str(w_shape):<20} {n_params:<12} ({act})")
            total_params += n_params
        print("-" * 55)
        print(f"  Total parameters: {total_params:,}")
        print("=" * 55)


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CREATE AND INSPECT NETWORK                                                 ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

nn_model = NeuralNetwork([8, 64, 32, 1], activation="relu")
nn_model.summary()

# Quick forward pass test
test_out = nn_model.forward(X_train[:3])
print(f"\nTest forward pass (3 samples): {test_out.flatten().round(4)}")


> **For your report:** How many total parameters does the [8, 64, 32, 1] network have? Explain He initialization — why scale weights by sqrt(2/n_in)?

---

# Part 8: Loss Function — Mean Squared Error

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  MSE LOSS FROM SCRATCH                                                      ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

def mse_loss(y_pred, y_true):
    """Mean Squared Error: L = (1/n) * sum((y_pred - y_true)^2)"""
    return np.mean((y_pred - y_true) ** 2)

# --- Demonstrate on simple examples ---
y_true_ex = np.array([1.0, 2.0, 3.0, 4.0, 5.0])
y_perfect = np.array([1.0, 2.0, 3.0, 4.0, 5.0])
y_close   = np.array([1.1, 2.2, 2.8, 4.3, 4.9])
y_bad     = np.array([3.0, 1.0, 5.0, 2.0, 4.0])

print(f"MSE (perfect predictions): {mse_loss(y_perfect, y_true_ex):.4f}")
print(f"MSE (close predictions):   {mse_loss(y_close, y_true_ex):.4f}")
print(f"MSE (bad predictions):     {mse_loss(y_bad, y_true_ex):.4f}")

# --- Initial loss with random weights (should be high) ---
y_pred_init = nn_model.forward(X_train)
initial_loss = mse_loss(y_pred_init, y_train)
print(f"\nInitial loss (random weights): {initial_loss:.4f}")
print("This should be high — the network hasn't learned anything yet!")


> **For your report:** Why is MSE a good loss function for regression? What does a high initial loss tell us about random weight initialization?

---

# Part 9: Gradient Descent (1D Demonstration)

Before applying gradient descent to our full network, let's visualize it on a simple 1D problem.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  1D GRADIENT DESCENT DEMO                                                   ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

# Simple loss function: L(w) = (w - 3)^2  (minimum at w=3)
def loss_1d(w):
    return (w - 3.0) ** 2

def grad_1d(w):
    return 2.0 * (w - 3.0)

# Run gradient descent with different learning rates
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
learning_rates = [0.01, 0.1, 0.9]
w_range = np.linspace(-2, 8, 200)

for ax, lr in zip(axes, learning_rates):
    ax.plot(w_range, loss_1d(w_range), "b-", linewidth=2, label="L(w) = (w-3)²")

    # Run GD
    w = -1.0  # start far from minimum
    path_w = [w]
    path_l = [loss_1d(w)]

    for step in range(30):
        grad = grad_1d(w)
        w = w - lr * grad
        path_w.append(w)
        path_l.append(loss_1d(w))

    ax.plot(path_w, path_l, "ro-", markersize=4, linewidth=1, label="GD path")
    ax.plot(path_w[0], path_l[0], "gs", markersize=10, label="Start")
    ax.plot(3, 0, "r*", markersize=15, label="Minimum")
    ax.set_title(f"lr = {lr}", fontweight="bold")
    ax.set_xlabel("w")
    ax.set_ylabel("Loss")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle("Gradient Descent with Different Learning Rates", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("asgn_fig_gradient_descent_1d.png", dpi=150, bbox_inches="tight")
plt.show()


> **For your report:** What happens with a very small learning rate (0.01)? A good one (0.1)? A too-large one (0.9)? How would you choose the right learning rate?

---

# Part 10: Backpropagation

Backpropagation computes the gradient of the loss with respect to every weight using the **chain rule**.

## Chain Rule for One Layer

For a layer with output a = f(W·x + b) and loss L:

| Gradient | Formula | Meaning |
|---|---|---|
| ∂L/∂a | from next layer (or loss derivative for output) | How much the loss changes with the activation |
| ∂L/∂z | ∂L/∂a · f'(z) | How much the loss changes with the pre-activation |
| ∂L/∂W | ∂L/∂z · xᵀ | How much the loss changes with the weights |
| ∂L/∂b | ∂L/∂z | How much the loss changes with the bias |
| ∂L/∂x | Wᵀ · ∂L/∂z | Gradient to pass to the previous layer |

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  GRADIENT CHECKING (verify backprop is correct)                             ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

def gradient_check(model, X_sample, y_sample, epsilon=1e-5):
    """
    Compare analytical gradients (from backprop) with numerical gradients.
    If they match, backprop is implemented correctly!
    """
    # Get analytical gradients
    model.forward(X_sample)
    dW_analytical, db_analytical = model.backward(y_sample)

    print("Gradient Checking (analytical vs numerical):")
    print("=" * 60)

    max_diff = 0.0

    for layer_idx in range(model.n_layers):
        # Check a few weights in each layer
        W = model.weights[layer_idx]
        dW_ana = dW_analytical[layer_idx]

        for i in range(min(3, W.shape[0])):
            for j in range(min(3, W.shape[1])):
                # Numerical gradient: (L(w+eps) - L(w-eps)) / (2*eps)
                old_val = W[i, j]

                W[i, j] = old_val + epsilon
                y_plus = model.forward(X_sample)
                loss_plus = model.compute_loss(y_plus, y_sample)

                W[i, j] = old_val - epsilon
                y_minus = model.forward(X_sample)
                loss_minus = model.compute_loss(y_minus, y_sample)

                W[i, j] = old_val  # restore

                grad_numerical = (loss_plus - loss_minus) / (2 * epsilon)
                grad_analytical = dW_ana[i, j]

                diff = abs(grad_numerical - grad_analytical)
                max_diff = max(max_diff, diff)

        print(f"  Layer {layer_idx + 1}: max diff = {max_diff:.2e}")

    print(f"\nMax absolute difference: {max_diff:.2e}")
    if max_diff < 1e-5:
        print("PASSED! Backprop gradients match numerical gradients.")
    else:
        print("WARNING: Large gradient mismatch. Check backprop implementation.")

# Run gradient check on a small batch
check_nn = NeuralNetwork([8, 16, 8, 1], activation="relu")
gradient_check(check_nn, X_train[:10], y_train[:10])


> **For your report:** Explain the chain rule for one layer of the network. Show the math for how ∂L/∂W is computed from ∂L/∂a and f'(z).

---

# Part 11: Training the Network

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  TRAIN THE NETWORK                                                          ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

import time

# Create fresh model
model = NeuralNetwork([8, 64, 32, 1], activation="relu")
model.summary()

# Hyperparameters
LEARNING_RATE = 0.01
EPOCHS = 200
BATCH_SIZE = 64

print(f"\nTraining with lr={LEARNING_RATE}, epochs={EPOCHS}, batch_size={BATCH_SIZE}")
print("-" * 60)

start_time = time.time()
history = model.train(X_train, y_train, X_test, y_test,
                      lr=LEARNING_RATE, epochs=EPOCHS,
                      batch_size=BATCH_SIZE, verbose=True)
elapsed = time.time() - start_time
print(f"\nTraining time: {elapsed:.1f}s")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  PLOT TRAINING CURVES                                                       ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

fig, ax = plt.subplots(1, 1, figsize=(10, 5))
ax.plot(history["train_loss"], label="Training Loss", linewidth=2)
ax.plot(history["val_loss"], label="Validation Loss", linewidth=2, linestyle="--")
ax.set_xlabel("Epoch")
ax.set_ylabel("MSE Loss")
ax.set_title("Training and Validation Loss Curves", fontweight="bold")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("asgn_fig_training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Final train loss: {history['train_loss'][-1]:.6f}")
print(f"Final val loss:   {history['val_loss'][-1]:.6f}")


> **For your report:** Interpret the loss curves. Is the model converging? Is there a gap between training and validation loss (overfitting)? At what epoch does the loss plateau?

---

# Part 12: Evaluation

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  TEST SET EVALUATION                                                        ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

# Predictions (in normalized space)
y_pred_norm = model.forward(X_test)

# Convert back to original scale
y_pred_orig = y_pred_norm.flatten() * y_std + y_mean
y_test_orig = y_test.flatten() * y_std + y_mean

# Metrics (from scratch)
mse  = np.mean((y_pred_orig - y_test_orig) ** 2)
rmse = np.sqrt(mse)
mae  = np.mean(np.abs(y_pred_orig - y_test_orig))
ss_res = np.sum((y_test_orig - y_pred_orig) ** 2)
ss_tot = np.sum((y_test_orig - np.mean(y_test_orig)) ** 2)
r2 = 1 - ss_res / ss_tot

print("=" * 50)
print("  Test Set Evaluation (original scale)")
print("=" * 50)
print(f"  MSE:  {mse:.4f}")
print(f"  RMSE: {rmse:.4f}  (in $100K)")
print(f"  MAE:  {mae:.4f}  (in $100K)")
print(f"  R²:   {r2:.4f}")
print("=" * 50)
print(f"\n  Average house price: ${np.mean(y_test_orig)*100000:.0f}")
print(f"  Average error:       ${mae*100000:.0f}")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  PREDICTED VS ACTUAL + RESIDUALS                                            ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter: predicted vs actual
ax = axes[0]
ax.scatter(y_test_orig, y_pred_orig, alpha=0.15, s=5, color="steelblue")
lims = [min(y_test_orig.min(), y_pred_orig.min()),
        max(y_test_orig.max(), y_pred_orig.max())]
ax.plot(lims, lims, "r--", linewidth=2, label="Perfect prediction")
ax.set_xlabel("Actual Price ($100K)")
ax.set_ylabel("Predicted Price ($100K)")
ax.set_title(f"Predicted vs Actual (R² = {r2:.4f})", fontweight="bold")
ax.legend()
ax.grid(True, alpha=0.3)

# Residual distribution
ax = axes[1]
residuals = y_pred_orig - y_test_orig
ax.hist(residuals, bins=50, alpha=0.7, color="steelblue", edgecolor="white")
ax.axvline(0, color="red", linewidth=2, linestyle="--")
ax.set_xlabel("Residual (Predicted - Actual)")
ax.set_ylabel("Count")
ax.set_title("Residual Distribution", fontweight="bold")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("asgn_fig_evaluation.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  BEST AND WORST PREDICTIONS                                                 ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

abs_errors = np.abs(residuals)
best_5 = np.argsort(abs_errors)[:5]
worst_5 = np.argsort(abs_errors)[-5:][::-1]

print("Top 5 BEST Predictions:")
print(f"  {'Actual':>10s}  {'Predicted':>10s}  {'Error':>10s}")
print("  " + "-" * 35)
for idx in best_5:
    print(f"  ${y_test_orig[idx]*100000:>9,.0f}  ${y_pred_orig[idx]*100000:>9,.0f}  ${residuals[idx]*100000:>+9,.0f}")

print(f"\nTop 5 WORST Predictions:")
print(f"  {'Actual':>10s}  {'Predicted':>10s}  {'Error':>10s}")
print("  " + "-" * 35)
for idx in worst_5:
    print(f"  ${y_test_orig[idx]*100000:>9,.0f}  ${y_pred_orig[idx]*100000:>9,.0f}  ${residuals[idx]*100000:>+9,.0f}")


> **For your report:** Discuss the model performance. What R² did you achieve? Are the residuals normally distributed? What patterns do you see in the best/worst predictions?

---

# Part 13: Hyperparameter Experiments

We systematically vary one hyperparameter at a time while keeping others fixed.

| Experiment | Variable | Values to Test |
|---|---|---|
| **A** | Architecture | [8,32,1], [8,64,32,1], [8,128,64,32,1] |
| **B** | Learning Rate | 0.0001, 0.001, 0.01, 0.1 |
| **C** | Activation | relu, sigmoid, tanh |

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  QUICK TRAIN HELPER                                                         ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

def quick_train(layer_sizes, activation="relu", lr=0.01, epochs=100, batch_size=64):
    """Train a model quickly and return its history and final metrics."""
    np.random.seed(SEED)
    net = NeuralNetwork(layer_sizes, activation=activation)
    history = net.train(X_train, y_train, X_test, y_test,
                        lr=lr, epochs=epochs, batch_size=batch_size, verbose=False)

    y_pred = net.forward(X_test)
    y_p = y_pred.flatten() * y_std + y_mean
    y_t = y_test.flatten() * y_std + y_mean

    rmse = np.sqrt(np.mean((y_p - y_t) ** 2))
    ss_res = np.sum((y_t - y_p) ** 2)
    ss_tot = np.sum((y_t - np.mean(y_t)) ** 2)
    r2 = 1 - ss_res / ss_tot

    return history, rmse, r2


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  EXPERIMENT A: ARCHITECTURE                                                 ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

architectures = {
    "Shallow [8,32,1]":       [8, 32, 1],
    "Medium [8,64,32,1]":     [8, 64, 32, 1],
    "Deep [8,128,64,32,1]":   [8, 128, 64, 32, 1],
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
results_arch = {}

for name, arch in architectures.items():
    hist, rmse_val, r2_val = quick_train(arch, lr=0.01, epochs=100)
    results_arch[name] = {"rmse": rmse_val, "r2": r2_val}
    axes[0].plot(hist["train_loss"], label=f"{name} (train)")
    axes[1].plot(hist["val_loss"], label=f"{name} (val)")

axes[0].set_title("Training Loss", fontweight="bold")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("MSE"); axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[1].set_title("Validation Loss", fontweight="bold")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("MSE"); axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle("Experiment A: Architecture Comparison", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("asgn_fig_exp_architecture.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nArchitecture Results:")
print(f"  {'Architecture':<25s}  {'RMSE':>8s}  {'R²':>8s}")
print("  " + "-" * 45)
for name, res in results_arch.items():
    print(f"  {name:<25s}  {res['rmse']:8.4f}  {res['r2']:8.4f}")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  EXPERIMENT B: LEARNING RATE                                                ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

learning_rates = [0.0001, 0.001, 0.01, 0.1]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
results_lr = {}

for lr_val in learning_rates:
    name = f"lr={lr_val}"
    hist, rmse_val, r2_val = quick_train([8, 64, 32, 1], lr=lr_val, epochs=100)
    results_lr[name] = {"rmse": rmse_val, "r2": r2_val}
    axes[0].plot(hist["train_loss"], label=f"{name} (train)")
    axes[1].plot(hist["val_loss"], label=f"{name} (val)")

axes[0].set_title("Training Loss", fontweight="bold")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("MSE"); axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[1].set_title("Validation Loss", fontweight="bold")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("MSE"); axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle("Experiment B: Learning Rate Comparison", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("asgn_fig_exp_learning_rate.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nLearning Rate Results:")
print(f"  {'LR':<15s}  {'RMSE':>8s}  {'R²':>8s}")
print("  " + "-" * 35)
for name, res in results_lr.items():
    print(f"  {name:<15s}  {res['rmse']:8.4f}  {res['r2']:8.4f}")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  EXPERIMENT C: ACTIVATION FUNCTION                                          ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

activations_exp = ["relu", "sigmoid", "tanh"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
results_act = {}

for act_name in activations_exp:
    hist, rmse_val, r2_val = quick_train([8, 64, 32, 1], activation=act_name, lr=0.01, epochs=100)
    results_act[act_name] = {"rmse": rmse_val, "r2": r2_val}
    axes[0].plot(hist["train_loss"], label=f"{act_name} (train)")
    axes[1].plot(hist["val_loss"], label=f"{act_name} (val)")

axes[0].set_title("Training Loss", fontweight="bold")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("MSE"); axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[1].set_title("Validation Loss", fontweight="bold")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("MSE"); axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle("Experiment C: Activation Function Comparison", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("asgn_fig_exp_activation.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nActivation Function Results:")
print(f"  {'Activation':<12s}  {'RMSE':>8s}  {'R²':>8s}")
print("  " + "-" * 35)
for name, res in results_act.items():
    print(f"  {name:<12s}  {res['rmse']:8.4f}  {res['r2']:8.4f}")


> **For your report:** For each experiment, state which setting performed best and explain WHY. Consider: model capacity, convergence speed, gradient flow, and computational cost.

---

# Part 14: Save & Conclusion

## 14.1 Save Model Weights

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  SAVE MODEL WEIGHTS                                                         ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

save_data = {
    "layer_sizes": model.layer_sizes,
    "activation": model.act_name,
    "weights": [w.tolist() for w in model.weights],
    "biases": [b.tolist() for b in model.biases],
    "normalization": {
        "X_mean": X_mean.tolist(),
        "X_std": X_std.tolist(),
        "y_mean": float(y_mean),
        "y_std": float(y_std),
    },
    "metrics": {"mse": float(mse), "rmse": float(rmse), "mae": float(mae), "r2": float(r2)},
}

import json
with open("nn_from_scratch_model.json", "w") as f:
    json.dump(save_data, f)

print("Model saved to: nn_from_scratch_model.json")
print(f"R² score: {r2:.4f}")


## 14.2 Summary of All Saved Figures

Every figure generated by this notebook is saved as a PNG for your report:

| Figure File | Content | Report Section |
|---|---|---|
| `asgn_fig_feature_distributions.png` | Histogram of all 9 variables | Dataset |
| `asgn_fig_correlations.png` | Scatter plots: feature vs target with correlation | Dataset |
| `asgn_fig_activations.png` | ReLU, Sigmoid, Tanh and their derivatives | Methodology |
| `asgn_fig_gradient_descent_1d.png` | 1D gradient descent with 3 learning rates | Methodology |
| `asgn_fig_training_curves.png` | Training and validation loss over epochs | Training |
| `asgn_fig_evaluation.png` | Predicted vs actual scatter + residual histogram | Results |
| `asgn_fig_exp_architecture.png` | Experiment A: architecture comparison | Experiments |
| `asgn_fig_exp_learning_rate.png` | Experiment B: learning rate comparison | Experiments |
| `asgn_fig_exp_activation.png` | Experiment C: activation function comparison | Experiments |

## 14.3 Report Writing Guide

### What to Write in Each Section

**1. Introduction (1 page)**
- What is a neural network? Explain neurons, layers, weights, biases, activation functions
- Why do we need non-linearity? What happens without it?
- Describe the forward pass, loss function, gradient descent, and backpropagation
- Include the architecture diagram from Part 1

**2. Dataset (0.5 page)**
- Describe California Housing: 20,640 samples, 8 features, target
- Include `asgn_fig_feature_distributions.png` and `asgn_fig_correlations.png`
- Explain preprocessing: z-score normalization (why needed?)

**3. Methodology (1 page)**
- Network architecture: [8, 64, 32, 1] with ReLU activation
- He initialization (explain why)
- MSE loss function (explain why for regression)
- Mini-batch SGD training (batch size, learning rate, epochs)
- Include the activation function plots

**4. Training (0.5 page)**
- Include `asgn_fig_training_curves.png`
- Discuss convergence speed and train/val gap

**5. Results (1 page)**
- Include all metrics: MSE, RMSE, MAE, R²
- Include `asgn_fig_evaluation.png`
- Discuss best/worst predictions

**6. Experiments (1 page)**
- For EACH experiment: include the comparison plot and results table
- State your observation and explain WHY
- Recommend the best setting for each hyperparameter

**7. Conclusion (0.5 page)**
- Summary of key findings
- What you learned about building neural networks from scratch
- Limitations and potential improvements

---

<center>

### Assignment Complete

**Remember to:**
1. Run all cells from top to bottom before submission
2. Ensure all outputs and figures are visible
3. Write your report using the generated figures
4. Save your model file

---

*Apeiron AI | "Boundless Possibilities, Infinite Potential"*
*© 2026 | www.aperionaiml.com*

</center>